In [1]:
%pip install pandas sqlalchemy pyodbc

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus
import pyodbc

In [7]:
print(pyodbc.drivers())

['SQL Server', 'ODBC Driver 17 for SQL Server', 'SQL Server Native Client RDA 11.0', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)']


In [4]:
servidor = "DESKTOP-KD5U4CT"
base_origen = "AdventureWorks2022"
base_stage = "StageDW"

driver = "ODBC Driver 17 for SQL Server"


conexion_origen = quote_plus(
    f"DRIVER={{{driver}}};"
    f"SERVER={servidor};"
    f"DATABASE={base_origen};"
    f"Trusted_Connection=yes;"
    f"TrustServerCertificate=yes;"
)

engine_origen = create_engine(
    f"mssql+pyodbc:///?odbc_connect={conexion_origen}"
)

In [3]:
usuario = 'marcelo'
password = '12345'
servidor = "DESKTOP-KD5U4CT"
base_origen = "AdventureWorks2022"


driver = "ODBC Driver 17 for SQL Server"

cadena = quote_plus(
    f"DRIVER={{{driver}}};"
    f"SERVER={servidor};"
    f"DATABASE={base_stage};"
    f"UID={usuario};"
    f"PWD={password};"
    f"TrustServerCertificate=yes;"
)

origen_engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={cadena}"
)

Engine(mssql+pyodbc:///?odbc_connect=DRIVER%3D%7BODBC+Driver+17+for+SQL+Server%7D%3BSERVER%3DDESKTOP-KD5U4CT%3BDATABASE%3DStageDW%3BUID%3Dmarcelo%3BPWD%3D12345%3BTrustServerCertificate%3Dyes%3B)

In [13]:
consulta = """
SELECT
    ProductID,
    Name,
    ProductNumber,
    Color,
    StandardCost,
    ListPrice
FROM Production.Product
"""

df = pd.read_sql(consulta, engine_origen)

print("Datos extraídos:")
print(df.head())

print("\nCantidad de registros:", len(df))

Datos extraídos:
   ProductID                   Name ProductNumber Color  StandardCost  \
0          1        Adjustable Race       AR-5381  None           0.0   
1          2           Bearing Ball       BA-8327  None           0.0   
2          3        BB Ball Bearing       BE-2349  None           0.0   
3          4  Headset Ball Bearings       BE-2908  None           0.0   
4        316                  Blade       BL-2036  None           0.0   

   ListPrice  
0        0.0  
1        0.0  
2        0.0  
3        0.0  
4        0.0  

Cantidad de registros: 504


In [ ]:

# ---------------------------------------------------------
# TRANSFORM
# ---------------------------------------------------------

df = df.rename(
    columns={
        "ProductID": "producto_id",
        "Name": "nombre_producto",
        "ProductNumber": "codigo_producto",
        "Color": "color",
        "StandardCost": "costo_estandar",
        "ListPrice": "precio_lista"
    }
)

# El Stage puede reemplazar valores faltantes simples
# para facilitar ejercicios posteriores de calidad.
df["color"] = df["color"].fillna("SIN COLOR")



In [ ]:

# ---------------------------------------------------------
# Conexión a StageDW
# ---------------------------------------------------------

conexion_stage = quote_plus(
    f"DRIVER={{{driver}}};"
    f"SERVER={servidor};"
    f"DATABASE={base_stage};"
    f"Trusted_Connection=yes;"
    f"TrustServerCertificate=yes;"
)

engine_stage = create_engine(
    f"mssql+pyodbc:///?odbc_connect={conexion_stage}"
)


# ---------------------------------------------------------
# LOAD
# ---------------------------------------------------------

df.to_sql(
    name="stage_producto",
    con=engine_stage,
    schema="dbo",
    if_exists="replace",
    index=False
)

print("\nCarga terminada correctamente.")